# Bagging intuition with three decision trees

A courier wants routine parcels to move quickly. **Auto** goes straight to dispatch. **Review** sends a parcel to a person for a closer check. We have 12 historical records and one new parcel to route.

**Draw three trays of records → train a tree on each tray → trace the same new parcel → combine the answers.**

**Before teaching.** In Colab, choose **Runtime → Run all**. Start at **Draw rows with replacement**. Open the cards under each coding step to show the result one piece at a time. The short training cells remain visible so you can walk through the Python. Each section starts with a reminder for anyone joining midway.

The records are synthetic and the seed keeps the results repeatable. The visuals are built from the fitted models with HTML and SVG in the notebook itself.

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

print(f'NumPy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}')

In [ ]:
# @title Load notebook visuals { display-mode: "form" }
from html import escape
from IPython.display import HTML, display

visual_style = '''
<style>
  .parcel-visual { color:#253747; background:#f4f8f7; border:1px solid #c9dad7; border-radius:16px;
    padding:clamp(12px, 3vw, 24px); font:16px/1.5 system-ui,sans-serif; box-sizing:border-box; }
  .parcel-visual h3 { margin:0 0 6px; font-size:clamp(20px, 4vw, 29px); color:#163c3b; }
  .parcel-visual p { margin:6px 0 13px; }
  .parcel-visual figure { margin:0; }
  .parcel-visual details { margin:8px 0 0; border:1px solid #c9dad7; background:#fff; border-radius:10px; }
  .parcel-visual summary { cursor:pointer; padding:10px 14px; font-weight:700; color:#163c3b; }
  .parcel-visual summary:focus-visible { outline:2px solid #117069; outline-offset:2px; border-radius:8px; }
  .parcel-visual details[open] > summary { background:#e5f1ee; border-radius:9px 9px 0 0; }
  .parcel-visual .visual-body { padding:12px 14px; }
  .parcel-visual .visual-tickets { display:flex; flex-wrap:wrap; gap:6px; margin:9px 0; }
  .parcel-visual .visual-ticket { min-width:35px; padding:5px 7px; border:1px solid #b5c5c6;
    border-radius:6px; color:#183e3b; background:#e8f2ee; text-align:center; font-weight:700; }
  .parcel-visual .visual-repeat { background:#fff0c4; border-color:#96691c; color:#603d04; }
  .parcel-visual .visual-omitted { background:#f5f7f8; border-style:dashed; color:#4a5964; }
  .parcel-visual figcaption { color:#364e5e; margin:6px 0; }
  .parcel-visual .visual-tree-shell { overflow:auto; }
  .parcel-visual .visual-tree { display:block; width:100%; max-width:820px; min-width:570px; margin:auto; }
  .parcel-visual .visual-cards { display:grid; grid-template-columns:repeat(auto-fit,minmax(min(100%,225px),1fr)); gap:10px; }
  .parcel-visual .visual-card { padding:12px; border:1px solid #c9dad7; border-radius:10px; background:#fff; }
  .parcel-visual .visual-card strong { color:#155854; font-size:17px; }
  .parcel-visual .visual-grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(54px,1fr)); gap:5px; }
  .parcel-visual .visual-tree-chip { border:1px solid #b5c5c6; border-radius:7px; text-align:center;
    padding:4px; font-size:12px; background:#f7f9f9; color:#42515b; }
  .parcel-visual .visual-tree-chip.in-mean { background:#d9f0eb; color:#133e3c; border-color:#62968d; }
  .parcel-visual .visual-tree-chip b { display:block; font-size:14px; }
</style>
'''

def show_visual(key, title, caption, body):
    display(HTML(f'{visual_style}<section class="parcel-visual" data-visual="{key}">'
                 f'<h3>{title}</h3><p>{caption}</p>{body}</section>'))

def visual_option(group, state, title, body, opened=False):
    is_open = ' open' if opened else ''
    return (f'<details name="{group}" data-state="{state}"{is_open}>'
            f'<summary>{title}</summary><div class="visual-body">{body}</div></details>')

def visual_tickets(ids, *, duplicates=False, omitted=False):
    seen = set()
    tickets = []
    for row_id in ids:
        repeat = duplicates and row_id in seen
        variant = ' visual-omitted' if omitted else ' visual-repeat' if repeat else ''
        note = ' (repeat)' if repeat else ''
        tickets.append(f'<span class="visual-ticket{variant}">{escape(row_id)}{note}</span>')
        seen.add(row_id)
    return '<div class="visual-tickets">' + ''.join(tickets) + '</div>'

def tree_visual(model, point):
    fitted = model.tree_
    path_nodes = set(model.decision_path(point).indices)
    feature_labels = ['Weight', 'Scan score']
    svg_edges, svg_nodes = [], []

    def visit(node, x, depth, half_span):
        y = 30 + depth * 102
        left, right = fitted.children_left[node], fitted.children_right[node]
        on_path = node in path_nodes
        if left == right:
            label = names[int(model.classes_[fitted.value[node, 0].argmax()])]
        else:
            label = f'{feature_labels[fitted.feature[node]]} ≤ {fitted.threshold[node]:.2f}'
        count = fitted.weighted_n_node_samples[node]
        node_role = 'draw' if count == 1 else 'draws'
        stroke = '#177669' if on_path else '#9fb1b5'
        fill = '#d9f0eb' if on_path else '#fff'
        svg_nodes.append(
            f'<rect x="{x - 68}" y="{y}" width="136" height="51" rx="8" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{2.8 if on_path else 1.3}"/>'
            f'<text x="{x}" y="{y + 20}" font-size="15" font-weight="700">{escape(label)}</text>'
            f'<text x="{x}" y="{y + 39}" font-size="13">{count:g} {node_role}</text>'
        )
        if left != right:
            for child, direction, answer in ((left, -1, 'Yes'), (right, 1, 'No')):
                child_x = x + direction * half_span
                stroke = '#177669' if child in path_nodes else '#9fb1b5'
                svg_edges.append(
                    f'<line x1="{x}" y1="{y + 51}" x2="{child_x}" y2="{y + 102}" '
                    f'stroke="{stroke}" stroke-width="{3.8 if child in path_nodes else 1.6}"/>'
                    f'<text x="{(x + child_x) / 2}" y="{y + 70}" font-size="13"'
                    f' stroke="#f4f8f7" stroke-width="4" paint-order="stroke">{answer}</text>'
                )
                visit(child, child_x, depth + 1, half_span / 2)

    visit(0, 330, 0, 162)
    height = 93 + fitted.max_depth * 102
    prediction = names[int(model.predict(point)[0])]
    by_class = dict(zip(model.classes_, model.predict_proba(point)[0]))
    p_review = by_class.get(1, 0.0)
    description = f'The new parcel reaches the {prediction} leaf. Review probability {p_review:.1%}.'
    return (f'<figure><div class="visual-tree-shell"><svg class="visual-tree" '
            f'viewBox="0 0 660 {height}" role="img" aria-label="{escape(description)}" '
            f'text-anchor="middle" fill="#253747" xmlns="http://www.w3.org/2000/svg">'
            f'{"".join(svg_edges + svg_nodes)}</svg></div><figcaption>'
            f'<strong>{prediction}</strong> · {p_review:.1%} Review probability. '
            f'The thick green path follows the new parcel. The node counts are training draws.</figcaption></figure>')

## Set up the parcel desk

We need one route for the parcel arriving now. Its weight is **5.65 kg** and its scan score is **5.70**. Higher scan scores indicate a stronger scanner warning. The past records tell us what the desk did for each of the 12 earlier parcels.

In the code, `X` contains the two measurements, `y` contains the recorded route, and `P1` through `P12` are ticket IDs we can follow through the draws.

In [ ]:
training = pd.DataFrame([
    ('P1', 1.2, 2.0, 0), ('P2', 2.0, 5.0, 0), ('P3', 2.5, 7.8, 0),
    ('P4', 3.5, 4.0, 0), ('P5', 4.0, 7.0, 1), ('P6', 5.0, 5.0, 0),
    ('P7', 6.1, 5.5, 1), ('P8', 7.0, 3.0, 0), ('P9', 7.5, 8.0, 1),
    ('P10', 8.2, 6.6, 1), ('P11', 8.8, 4.5, 1), ('P12', 9.2, 8.8, 1),
], columns=['parcel_id', 'weight', 'scan_score', 'label']).set_index('parcel_id')

features = ['weight', 'scan_score']
X = training[features]
y = training['label']
names = {0: 'Auto', 1: 'Review'}
parcel = pd.DataFrame([(5.65, 5.70)], columns=features, index=['New parcel'])
bootstrap_seed = 11
tree_seed = 0

display(training[features].assign(route=y.map(names)), parcel)

## Draw rows with replacement

Imagine that each of our 12 past parcels has a ticket. Pick one at random, copy its whole record into Tree 1’s training tray, and return the original ticket to the desk. Because the original goes back, any parcel can come up on any draw. After 12 draws, that tray is one **bootstrap sample**.

In the code, `rng.choice` picks row positions and `iloc` uses them to select records. The seed makes the same draw sequence easy to replay.

**Try the visual below.** Open **1 draw**, then ask whether the same ticket can come up next. Open **2 draws** to check. Open **All 12 draws** to find the omitted IDs. The **Without replacement** card shows a separate comparison, so the original tray stays ready for the next coding cell.

In [ ]:
rng = np.random.default_rng(bootstrap_seed)
draw = rng.choice(len(X), size=len(X), replace=True)
drawn_ids = training.index[draw]
counts = drawn_ids.value_counts().reindex(training.index, fill_value=0)

print(f'{len(draw)} draws, {drawn_ids.nunique()} distinct rows')
print('Draw order:', ', '.join(drawn_ids))
print('Omitted:', ', '.join(counts.index[counts == 0]) or 'none')
display(counts.rename('times drawn').to_frame().T)

In [ ]:
# @title Explore the training tray { display-mode: "form" }
def draw_panel(ids, complete=False):
    body = '<strong>Draw order</strong>' + visual_tickets(ids.tolist(), duplicates=True)
    draw_count, parcel_count = len(ids), ids.nunique()
    draw_label = 'draw' if draw_count == 1 else 'draws'
    parcel_label = 'parcel' if parcel_count == 1 else 'parcels'
    body += f'<p>{draw_count} {draw_label} · {parcel_count} distinct {parcel_label}</p>'
    if complete:
        omitted = training.index.difference(ids, sort=False)
        if len(omitted):
            body += '<strong>Still on the desk, absent from this tray</strong>'
            body += visual_tickets(omitted.tolist(), omitted=True)
        else:
            body += '<p>Every parcel appears once. The order has changed, and the records are all still there.</p>'
    return body

shuffled_ids = training.index[
    np.random.default_rng(bootstrap_seed).choice(len(X), size=len(X), replace=False)
]
draw_cards = (
    visual_option('bootstrap-stages', 'draw-1', '1 draw', draw_panel(drawn_ids[:1]), opened=True)
    + visual_option('bootstrap-stages', 'draw-2', '2 draws', draw_panel(drawn_ids[:2]))
    + visual_option('bootstrap-stages', 'draw-all', 'All 12 draws', draw_panel(drawn_ids, complete=True))
    + visual_option('bootstrap-stages', 'no-replacement', 'Without replacement',
                    draw_panel(shuffled_ids, complete=True))
)
show_visual('bootstrap-draws', 'Build one training tray',
            'Copy the drawn record into the tray and return the original to the desk. '
            'Open a step to see what is in the tray.', draw_cards)

## Fit one tree and predict one parcel

So far we have one tray of training records. The tree sees the entire record each time it appears, including the route label. That means a repeat has another say when the tree chooses a cutoff.

This tree can ask at most two questions along any path. Look for the first question, then follow the thick green line for the new parcel. With its scan score of 5.70, which way does it go at the root?

In [ ]:
tree = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=tree_seed)
tree.fit(X.iloc[draw], y.iloc[draw])
prediction = int(tree.predict(parcel)[0])

print(f'Tree 1 predicts {names[prediction]} for the new parcel.')
print(export_text(tree, feature_names=features, decimals=2))

In [ ]:
# @title Trace one decision tree { display-mode: "form" }
show_visual('one-tree', 'One tray → one tree → one answer',
            'Follow the questions for our new parcel, which weighs 5.65 kg and has a scan score of 5.70.',
            tree_visual(tree, parcel))

## Bootstrap, fit, predict, then combine

The new parcel is still **5.65 kg, scan score 5.70**. We make a different tray for each of three trees by resampling the same original 12 records. Every tree uses the same learning settings, then gets the same new parcel.

The loop fits one tree per tray. Each model returns its probabilities in the order given by its `classes_`. We place those probabilities in matching columns. Open the fitted trees below and compare the question each one asks first.

In [ ]:
rng = np.random.default_rng(bootstrap_seed)
demo_draws = rng.choice(len(X), size=(3, len(X)), replace=True)
trees = []
probabilities = np.zeros((len(demo_draws), len(names)))
demo_results = []

for index, sample in enumerate(demo_draws):
    model = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=tree_seed)
    model.fit(X.iloc[sample], y.iloc[sample])
    trees.append(model)
    probabilities[index, model.classes_] = model.predict_proba(parcel)[0]
    demo_results.append({
        'tree': index + 1,
        'distinct rows': len(np.unique(sample)),
        'root feature': features[model.tree_.feature[0]],
        'root cut': model.tree_.threshold[0],
        'class': names[int(model.predict(parcel)[0])],
        'P(Review)': probabilities[index, 1],
    })

display(pd.DataFrame(demo_results).round(4))

In [ ]:
# @title Compare the three fitted trees { display-mode: "form" }
tree_cards = []
for index, (sample, model) in enumerate(zip(demo_draws, trees), start=1):
    ids = training.index[sample]
    tray = visual_tickets(ids.tolist(), duplicates=True)
    tree_cards.append(visual_option(
        'parcel-trees', f'tree-{index}', f'Tree {index}',
        f'<strong>Training tray · {ids.nunique()} distinct parcels</strong>{tray}'
        + tree_visual(model, parcel), opened=index == 1,
    ))

show_visual('manual-forest', 'Three trees see the same parcel',
            'Open a tree to see its training tray, the questions it learned, and its route for '
            'the 5.65 kg parcel with scan score 5.70.', ''.join(tree_cards))

## Two ways to aggregate a classification prediction

**Hard voting.** Each tree contributes its predicted class. Pick the class with the most votes.

**Probability averaging.** Each tree contributes its probability for each class. Average those numbers class by class and pick the class with the highest mean probability. These rules can yield different outcomes because they use different information. A tie below resolves to the smallest encoded class.

**Worked checkpoint.** In tree order, the predicted classes are Auto, Review, and Review. The corresponding Review probabilities are 0, 1, and 2/3. Review wins the hard vote 2 to 1 and also wins by the mean Review probability of 5/9, about 55.6%.

**Regression.** When trees predict a numerical target, bagging can average their numerical predictions directly.

In [ ]:
hard_predictions = np.array([model.predict(parcel)[0] for model in trees])
votes = np.bincount(hard_predictions, minlength=len(names))
mean_probabilities = probabilities.mean(axis=0)

print(f'Hard vote: Auto {votes[0]}, Review {votes[1]} -> {names[int(votes.argmax())]}')
print(f'Mean P(Review): {mean_probabilities[1]:.4f} -> {names[int(mean_probabilities.argmax())]}')
np.testing.assert_array_equal(hard_predictions, [0, 1, 1])
np.testing.assert_allclose(probabilities[:, 1], [0, 1, 2 / 3])

In [ ]:
# @title Compare the two ways to combine answers { display-mode: "form" }
vote_text = ' + '.join(names[int(prediction)] for prediction in hard_predictions)
leaf_fractions = []
for model, sample in zip(trees, demo_draws):
    leaf = model.apply(parcel)[0]
    in_leaf = model.apply(X.iloc[sample]) == leaf
    rows = y.iloc[sample].to_numpy()[in_leaf]
    leaf_fractions.append(f'{np.count_nonzero(rows == 1)}/{len(rows)}')
formula = ' + '.join(leaf_fractions)

show_visual('aggregation', 'Turn three answers into one route',
            'All three trees predict the same new parcel. Read across to compare giving each tree one vote '
            'with using the actual mix of training records in its leaf.',
            f'<div class="visual-cards"><div class="visual-card"><strong>Hard vote</strong>'
            f'<p>{vote_text}</p><p>{votes[0]} Auto · {votes[1]} Review → '
            f'<b>{names[int(votes.argmax())]}</b></p></div>'
            f'<div class="visual-card"><strong>Average Review probability</strong>'
            f'<p>({formula}) ÷ {len(trees)} = {mean_probabilities[1]:.1%}</p>'
            f'<p>Higher mean probability → <b>{names[int(mean_probabilities.argmax())]}</b></p></div></div>')

## Connect the example to Random Forest

The three-tree example built the trays and combined the predictions by hand. Now give those jobs to scikit-learn. A Random Forest with `bootstrap=True` samples one tray per tree. The extra ingredient is a new draw of candidate features at each split.

Here `max_features=1` asks the splitter to consider one candidate feature at a time, with additional candidates inspected if needed to find a valid split. Each tree returns a probability for each class. The forest averages the probabilities and picks the class with the highest mean.

With scikit-learn 1.6.1 the seeded 64-tree forest predicts Review with mean probability 0.6873. This forest makes its own draws, so it produces a different probability from the three-tree exercise. The squares below show the contributions from the actual fitted estimators.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=64, max_depth=2, max_features=1,
    bootstrap=True, random_state=bootstrap_seed, n_jobs=1,
).fit(X, y)

# This example encodes the two classes as 0 and 1, in the same column order.
parcel_array = parcel.to_numpy()
tree_probabilities = np.array([estimator.predict_proba(parcel_array)[0]
                               for estimator in forest.estimators_])
forest_probabilities = forest.predict_proba(parcel)[0]
np.testing.assert_allclose(tree_probabilities.mean(axis=0), forest_probabilities)
display(pd.Series(forest_probabilities, index=[names[int(c)] for c in forest.classes_],
                  name='Mean forest probability').round(4).to_frame().T)
print('Random Forest prediction:', names[int(forest.predict(parcel)[0])])

In [ ]:
# @title Explore the fitted Random Forest { display-mode: "form" }
def forest_panel(limit):
    chips = []
    for i, p_review in enumerate(tree_probabilities[:limit, 1], start=1):
        chips.append(
            f'<span class="visual-tree-chip in-mean" aria-label="Tree {i}, {p_review:.1%} Review">'
            f'Tree {i}<b>{p_review:.0%}</b></span>'
        )
    mean = tree_probabilities[:limit].mean(axis=0)
    route = names[int(forest.classes_[mean.argmax()])]
    body = (f'<p><strong>{limit} {"tree" if limit == 1 else "trees"} included</strong> · '
            f'Average Review probability {mean[1]:.1%} → <strong>{route}</strong></p>'
            f'<div class="visual-grid">{"".join(chips)}</div>')
    if limit == 1:
        body += tree_visual(forest.estimators_[0], parcel_array)
    return body

forest_cards = ''.join(
    visual_option('parcel-forest-sizes', f'forest-{limit}', title, forest_panel(limit), opened=limit == 1)
    for limit, title in ((1, 'First tree'), (8, 'First 8 trees'), (64, 'All 64 trees'))
)
show_visual('random-forest', 'Inside the fitted Random Forest',
            'Each tile shows one fitted tree’s Review probability for our new parcel. '
            'Open the next step to include more trees in the average.',
            forest_cards)